In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url='http://g02:9079/v1',
    api_key='dummy'
)
model_name = client.models.list().data[0].id
print(f'Using model: {model_name}')

# models = client.models.list()
# models = sorted(models.data, key=lambda m: m.id)
# for model in models:
#     print(model.id)

In [ ]:
test_response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(test_response.choices[0].message.content)

In [ ]:
test_schema = {"type": "object",
    "properties": {
        "name": {
        "type": "string"
        },
        "year of birth": {
        "type": ["integer", "null"]
        },
        "year of death": {
        "type": ["integer", "null"]
        }
    },
    "required": ["name"],
    "additionalProperties": False
}
completion = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": "Get info on Winston Churchil."
        }
    ],
    response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "answer",
                        "strict": True,
                        "schema": test_schema,
                    }
            },
)

print(completion.choices[0].message.content)

In [ ]:
import openai
from pydantic import BaseModel

class Pet(BaseModel):
    name: str
    animal: str
    age: int
    color: str | None
    favorite_toy: str | None

class PetList(BaseModel):
    pets: list[Pet]

try:
    completion = client.beta.chat.completions.parse(
        temperature=0,
        model=model_name,
        messages=[
            {"role": "user", "content": '''
                I have two pets.
                A cat named Luna who is 5 years old and loves playing with yarn. She has grey fur.
                I also have a 2 year old black cat named Loki who loves tennis balls.
            '''}
        ],
        response_format=PetList,
    )

    pet_response = completion.choices[0].message
    if pet_response.parsed:
        print(pet_response.parsed)
    elif pet_response.refusal:
        print(pet_response.refusal)
except Exception as e:
    if type(e) == openai.LengthFinishReasonError:
        print("Too many tokens: ", e)
        pass
    else:
        print(e)
        pass

print(pet_response.parsed.pets[0])

In [ ]:
from pydantic import BaseModel
from typing import Literal
import outlines
import openai

class Customer(BaseModel):
    name: str
    urgency: Literal["high", "medium", "low"]
    issue: str

# This is for vllm 0.12+
model = outlines.from_openai(client, model_name)

customer = model(
    "Alice needs help with login issues ASAP",
    Customer
)
print(customer) # This is a string representation
print(Customer.model_validate_json(customer)) # Pydantic model instance

# This is for vllm 0.8.5, it uses outlines 0.1.11
# from openai import AsyncOpenAI
# from outlines.models.openai import OpenAIConfig
# client = AsyncOpenAI(
#     api_key='dummy',
#     base_url='http://g09:9079/v1',
# )
# config = OpenAIConfig(model_name,
#                       presence_penalty=1.,
#                       frequency_penalty=1.,
#                       top_p=.95)
# ol = outlines.models.openai(
#     client,
#     config,
# )
# generate_customer = outlines.generate.json(ol, Customer)
# customer = generate_customer(
#     "Alice needs help with login issues ASAP",
# )
# print(customer) # Pydantic model instance
# print(customer.model_dump()) # Dictionary